# FRI Hash Computation Notebook

This notebook calculates the number of hash operations performed by the FRI prover and verifier based on input parameters.

## FRI Prover

### Commit

In [186]:
import math

def count_mmcs_commit_hashes(shapes):
    assert shapes, "shapes must not be empty"
    
    shapes.sort(reverse=True)
    
    index = 0
    count = 0
    
    width = 0
    height = shapes[index][0]

    while index < len(shapes) and shapes[index][0] == height:
        width += shapes[index][1]
        index += 1
    tmp_count = (height // 4 + height % 4) * ((width + 7) // 8)
    count += tmp_count
    
    cur_height = height
    
    while cur_height > 1:
        nxt_height = cur_height >> 1
        width = 0
        while index < len(shapes) and shapes[index][0] == nxt_height:
            width += shapes[index][1]
            index += 1
        tmp_count = (nxt_height // 4 + nxt_height % 4)
        if width > 0:
            tmp_count += (nxt_height // 4 + nxt_height % 4) * ((width + 7) // 8 + 1)
            
        count += tmp_count

        cur_height = nxt_height
        
    return count
 
# Testcases
assert count_mmcs_commit_hashes([(8, 2), (4, 3)]) == 8
assert count_mmcs_commit_hashes([(8, 5), (16, 3)]) == 14

In [187]:
def count_commit_base_hashes(shapes, log_blowup):
    """
    Compute the number of hashes when committing the witnesses.

    Args:
        shapes (list): The list of (height, width) pairs of `RowMajorMatrix`s.
        log_arity (int): The log of the arity of the Merkle tree.
        log_blow_up (int): The log of the blowup factor.
        log_final_len (int): The log of the length of the final polynomial.
    """
    
    shapes = [(h << log_blowup, w) for h, w in shapes]
    count_mmcs_commit_hashes(shapes)

In [188]:
def count_commit_extn_hashes(shapes, log_blowup):
    """
    Compute the number of hashes when committing the witnesses.

    Args:
        shapes (list): The list of (height, width) pairs of `RowMajorMatrix`s.
        log_arity (int): The log of the arity of the Merkle tree.
        log_blow_up (int): The log of the blowup factor.
        log_final_len (int): The log of the length of the final polynomial.
    """
    
    shapes = [(h << log_blowup, w * 4) for h, w in shapes]
    count_mmcs_commit_hashes(shapes)

### Prove

In [189]:
def count_prove_hashes(shapes, log_arity, log_blowup, log_final_len):
    """
    Compute the number of hashes when committing the witnesses.

    Args:
        shapes (list): The list of heights of `RowMajorMatrix`s.
        log_arity (int): The log of the arity of the Merkle tree.
        log_blow_up (int): The log of the blowup factor.
        log_final_len (int): The log of the length of the final polynomial.
    """
    assert shapes, "shapes must not be empty"
    
    # reduced shapes
    shapes = [h << log_blowup for h in shapes]
    shapes = list(set(shapes))
    shapes.sort(reverse=True)
    
    # The hashes only occur in the commit phase.
    cur_height = shapes[0]
    arity = 1 << log_arity
    count = 0
    index = 0
    while cur_height > 1 << (log_blowup + log_final_len):
        cur_arity = min(arity, cur_height)
        nxt_height = cur_height // cur_arity
        cur_shapes = [(nxt_height, 4 * cur_arity)]
        while index < len(shapes) and shapes[index] > nxt_height:
            if shapes[index] < cur_height:
                cur_shapes.append((nxt_height, 4 * shapes[index] // nxt_height))
            index += 1
        tmp_count = count_mmcs_commit_hashes(cur_shapes)
        count += tmp_count
        cur_height = nxt_height
        
    return count

# Testcases
assert count_prove_hashes([1 << 5, 1 << 6, 1 << 7], log_arity=1, log_blowup=1, log_final_len=0) == 141
assert count_prove_hashes([1 << 5, 1 << 6, 1 << 7], log_arity=2, log_blowup=1, log_final_len=0) == 87

### Verify

In [190]:
def count_mmcs_verify_hashes(shapes):
    assert shapes, "shapes must not be empty"
    
    shapes.sort(reverse=True)
    
    index = 0
    count = 0
    
    width = 0
    height = shapes[index][0]
    
    while index < len(shapes) and shapes[index][0] == height:
        width += shapes[index][1]
        index += 1
    tmp_count = ((width + 7) // 8)
    count += tmp_count
    cur_height = height
    
    while cur_height > 1:
        nxt_height = cur_height >> 1
        width = 0
        while index < len(shapes) and shapes[index][0] == nxt_height:
            width += shapes[index][1]
            index += 1
        tmp_count = 1
        if width > 0:
            tmp_count += ((width + 7) // 8 + 1)
        count += tmp_count

        cur_height = nxt_height
        
    return count
    

def count_verify_hashes(wit_batch_shapes, reduced_shapes, log_arity, log_blowup, log_final_len, n_queries):
    """
    Compute the number of hashes when committing the witnesses.

    Args:
        wit_batch_shapes (list(list)): The batches of (height, width) pairs of `RowMajorMatrix`s.
        reduced_shapes (list): The list of heights of the commit phase shapes.
        log_arity (int): The log of the arity of the Merkle tree.
        log_blow_up (int): The log of the blowup factor.
        log_final_len (int): The log of the length of the final polynomial.
        n_queries (int): The number of queries.
    """
    
    count_per_query = 0
    
    # verify batch
    for shapes in wit_batch_shapes:
        count_per_query += count_mmcs_verify_hashes(shapes, log_arity, log_blowup, log_final_len)
    
    # reduced shapes
    shapes = [h << log_blowup for h in reduced_shapes]
    shapes = list(set(shapes))
    shapes.sort(reverse=True)
    
    assert shapes, "shapes must not be empty"
    cur_height = shapes[0]
    arity = 1 << log_arity
    index = 0
    while cur_height > 1 << (log_blowup + log_final_len):
        cur_arity = min(arity, cur_height)
        nxt_height = cur_height // cur_arity
        cur_shapes = [(nxt_height, 4 * cur_arity)]
        while index < len(shapes) and shapes[index] > nxt_height:
            if shapes[index] < cur_height:
                cur_shapes.append((nxt_height, 4 * shapes[index] // nxt_height))
            index += 1
        tmp_count = count_mmcs_verify_hashes(cur_shapes)
        count_per_query += tmp_count
        cur_height = nxt_height
        
    return count_per_query * n_queries

# test cases
assert count_verify_hashes([], [1 << 5, 1 << 6, 1 << 7], log_arity=1, log_blowup=1, log_final_len=0, n_queries=1) == 35
assert count_verify_hashes([], [1 << 5, 1 << 6, 1 << 7], log_arity=2, log_blowup=1, log_final_len=0, n_queries=1) == 21